In [1]:
# 安装数据处理必备库
!pip install pandas jieba

# 如果要从OBS（对象存储）直接读写数据，还需要安装华为云的moxing库
# 新版本ModelArts中可能已预装，或更名为os或obs
#!pip install moxing --upgrade
!pip install huaweicloud-sdk-python-obs
# 验证MindSpore和Python版本是否与您的图片信息一致
import mindspore as ms
import sys
print("MindSpore Version:", ms.__version__)
print("Python Version:", sys.version)

Looking in indexes: http://pip.modelarts.private.com:8888/repository/pypi/simple

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Looking in indexes: http://pip.modelarts.private.com:8888/repository/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 13.1 MB/s eta 0:00:0000:0100:01
  Created wheel for huaweicloud-sdk-python-obs: filename=huaweicloud_sdk_python_obs-3.21.8-py3-none-any.whl size=87583 sha256=f3c76b97f8036552b4dbc64b8a7ec3799ccad63730a6a4af160087faadb15285
  Stored in directory: /home/ma-user/.cache/pip/wheels/1a/3f/86/8ea2e17ebc089a37349b0ce5087438011ac9d76eeea1bd9778
Successfully built huaweicloud-sdk-python-obs

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


MindSpore Version: 2.7.0
Python Version: 3.10.14 (main, Jun 25 2025, 09:03:25) [GCC 11.4.0]


In [4]:
ls


DeepSeek-R1-Distill-Qwen-1.5B/            kernel_meta/
Untitled.ipynb                            lost+found/
Untitled1.ipynb                           mindnlp/
chinese-poetry-master.zip                 offload/
deepseek-r1-distill-qwen-1_5b-lora.ipynb  output_1.5bf/
huanhuan.json


In [ ]:
!unzip chinese-poetry-master.zip

In [2]:
import os
import json
import pandas as pd
from tqdm import tqdm
import mindspore as ms
from mindspore.dataset import GeneratorDataset

# 1. 根据您的图片修正路径
repo_path = "./chinese-poetry-master/全唐诗"  # 根据您的图片修正路径
output_dir = "./processed_data"
os.makedirs(output_dir, exist_ok=True)

print(f"数据源: {repo_path}")
print(f"输出目录: {output_dir}")

# 2. 安装繁简转换库
try:
    import opencc
except ImportError:
    print("正在安装opencc繁简转换库...")
    !pip install opencc-python-reimplemented
    import opencc

# 初始化转换器
converter = opencc.OpenCC('t2s')  # 繁体转简体

def traditional_to_simplified(text):
    """将繁体中文转换为简体中文"""
    if text and isinstance(text, str):
        return converter.convert(text)
    return text

# 3. 根据您的实际文件选择目标文件
# 从您的图片中可以看到有 poet.song.X.json 文件
target_files = [
    "poet.song.0.json", "poet.song.1000.json",  # 宋词
    "唐诗三百首.json"  # 唐诗精选
]

# 4. 加载数据并进行繁简转换
def load_and_convert_data(files):
    """加载数据并转换为简体中文"""
    all_poems = []
    
    for file in tqdm(files, desc="加载和转换数据"):
        file_path = os.path.join(repo_path, file)  # 使用修正后的路径
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
                # 对每首诗进行繁简转换
                for poem in data:
                    # 转换各个字段
                    if 'title' in poem:
                        poem['title'] = traditional_to_simplified(poem['title'])
                    if 'author' in poem:
                        poem['author'] = traditional_to_simplified(poem['author'])
                    if 'dynasty' in poem:
                        poem['dynasty'] = traditional_to_simplified(poem.get('dynasty', ''))
                    
                    # 转换内容
                    if 'content' in poem:
                        poem['content'] = traditional_to_simplified(poem['content'])
                    
                    # 转换段落
                    if 'paragraphs' in poem:
                        poem['paragraphs'] = [traditional_to_simplified(p) for p in poem['paragraphs']]
                
                all_poems.extend(data)
                print(f"已处理 {file}: {len(data)} 首诗词")
                
        except Exception as e:
            print(f"处理文件 {file} 时出错: {e}")
    
    print(f"\n成功转换 {len(all_poems)} 首诗词到简体中文")
    return all_poems

# 执行数据加载和转换
simplified_poems = load_and_convert_data(target_files)

# 5. 保存转换后的简体数据
simplified_data_path = os.path.join(output_dir, "simplified_poetry_data.json")
with open(simplified_data_path, 'w', encoding='utf-8') as f:
    json.dump(simplified_poems, f, ensure_ascii=False, indent=2)
print(f"简体中文数据已保存: {simplified_data_path}")

# 6. 创建训练样本
def create_training_samples(poems):
    """创建用于模型训练的样本"""
    training_samples = []
    
    for poem in tqdm(poems, desc="创建训练样本"):
        title = poem.get('title', '无题')
        paragraphs = poem.get('paragraphs', [])
        content = ''.join(paragraphs) if paragraphs else poem.get('content', '')
        
        if not content or len(content) < 4:
            continue
            
        # 样本类型1: 续写任务
        if len(paragraphs) >= 2:
            training_samples.append({
                "instruction": "请续写下面的诗句",
                "input": paragraphs[0],
                "output": ''.join(paragraphs[1:]),
                "type": "续写"
            })
        
        # 样本类型2: 命题作诗
        training_samples.append({
            "instruction": "请根据题目创作一首诗",
            "input": f"题目：《{title}》",
            "output": content,
            "type": "命题"
        })
        
        # 样本类型3: 关键词作诗
        if len(content) >= 4:
            training_samples.append({
                "instruction": "请根据关键词创作一首诗",
                "input": f"关键词：{content[:4]}",
                "output": content,
                "type": "关键词"
            })
    
    print(f"共生成 {len(training_samples)} 个训练样本")
    return training_samples

# 创建训练样本
training_samples = create_training_samples(simplified_poems)

# 7. 保存训练数据
training_data_path = os.path.join(output_dir, "poetry_training_data.json")
with open(training_data_path, 'w', encoding='utf-8') as f:
    json.dump(training_samples, f, ensure_ascii=False, indent=2)
print(f"训练数据已保存: {training_data_path}")

# 8. 数据统计和验证
print("\n=== 数据统计 ===")
print(f"总诗词数量: {len(simplified_poems)}")
print(f"训练样本数量: {len(training_samples)}")

# 显示样本示例
print("\n=== 样本示例 ===")
for i, sample in enumerate(training_samples[:3]):
    print(f"\n示例 {i+1}:")
    print(f"类型: {sample['type']}")
    print(f"指令: {sample['instruction']}")
    print(f"输入: {sample['input']}")
    print(f"输出: {sample['output']}")
    print("-" * 50)

print("\n=== 预处理完成 ===")
print("下一步可以进行模型训练了!")

数据源: ./chinese-poetry-master/全唐诗
输出目录: ./processed_data
正在安装opencc繁简转换库...
Looking in indexes: http://pip.modelarts.private.com:8888/repository/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.8/481.8 kB 28.9 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


加载和转换数据:  33%|███▎      | 1/3 [00:00<00:00,  2.48it/s]

已处理 poet.song.0.json: 1000 首诗词


加载和转换数据: 100%|██████████| 3/3 [00:00<00:00,  3.26it/s]

已处理 poet.song.1000.json: 1000 首诗词
已处理 唐诗三百首.json: 366 首诗词

成功转换 2366 首诗词到简体中文


简体中文数据已保存: ./processed_data/simplified_poetry_data.json


创建训练样本: 100%|██████████| 2366/2366 [00:00<00:00, 7527.57it/s]

共生成 6911 个训练样本
训练数据已保存: ./processed_data/poetry_training_data.json

=== 数据统计 ===
总诗词数量: 2366
训练样本数量: 6911

=== 样本示例 ===

示例 1:
类型: 续写
指令: 请续写下面的诗句
输入: 欲出未出光辣达，千山万山如火发。
输出: 须臾走向天上来，逐却残星赶却月。
--------------------------------------------------

示例 2:
类型: 命题
指令: 请根据题目创作一首诗
输入: 题目：《日诗》
输出: 欲出未出光辣达，千山万山如火发。须臾走向天上来，逐却残星赶却月。
--------------------------------------------------

示例 3:
类型: 关键词
指令: 请根据关键词创作一首诗
输入: 关键词：欲出未出
输出: 欲出未出光辣达，千山万山如火发。须臾走向天上来，逐却残星赶却月。
--------------------------------------------------

=== 预处理完成 ===
下一步可以进行模型训练了!


In [3]:
# 在Notebook中运行这些命令来安装依赖
!pip install transformers
!pip install sentencepiece

Looking in indexes: http://pip.modelarts.private.com:8888/repository/pypi/simple

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Looking in indexes: http://pip.modelarts.private.com:8888/repository/pypi/simple

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [44]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# 指定模型所在的路径（就是当前目录，所以用 '.' 或者直接写文件夹名字都可以）
model_name = "./"  # 如果模型文件就在当前目录，用这个
# 或者如果文件在一个子文件夹里，比如 'DeepSeek-R1-Distill-Qwen-1.5B'，则用：
# model_name = "./DeepSeek-R1-Distill-Qwen-1.5B/"

# 加载分词器 (Tokenizer)
tokenizer = AutoTokenizer.from_pretrained('./DeepSeek-R1-Distill-Qwen-1.5B')

# 加载模型 (Model)
model = AutoModelForCausalLM.from_pretrained('./DeepSeek-R1-Distill-Qwen-1.5B')
# 将模型设置为评估模式（例如，用于推理/生成文本，而不是训练）
model.eval()

print("模型和分词器加载完成！")

模型和分词器加载完成！


In [13]:
# 使用模型进行文本生成示例
prompt = "中国的首都是"
inputs = tokenizer(prompt, return_tensors="pt")

# 生成文本
generate_ids = model.generate(**inputs, max_length=50)
response = tokenizer.batch_decode(generate_ids, skip_special_tokens=True)[0]

print(response)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


中国的首都是北京，那么北京的地理坐标是怎样的呢？我需要了解北京的位置和相关坐标，以便更好地理解中国的地理位置。
好的，现在我来思考一下北京的地理坐标。北京位于中国的一个城市，它


In [5]:
# 9. 转换为最终训练格式（纯文本）
final_training_file = os.path.join(output_dir, "poetry_train.txt")

with open(final_training_file, 'w', encoding='utf-8') as f:
    for sample in training_samples:
        # 这是一种简单的拼接方式，您可以根据需要设计更复杂的模板
        text_line = f"指令：{sample['instruction']}。输入：{sample['input']}。输出：{sample['output']}"
        # 或者使用更复杂的模板，例如：
        # text_line = f"<|im_start|>user\n{sample['instruction']}：{sample['input']}<|im_end|>\n<|im_start|>assistant\n{sample['output']}<|im_end|>"
        f.write(text_line + '\n')

print(f"最终训练文本已保存: {final_training_file}")

最终训练文本已保存: ./processed_data/poetry_train.txt


In [27]:
from datasets import load_dataset

# 路径指向您已经生成的结构化JSON文件，而不是那个.txt文件
final_training_file = './processed_data/poetry_training_data.json' # 关键修改！

# 使用 'json' 格式来加载结构化数据
raw_datasets = load_dataset('json', data_files=final_training_file)

# 从加载结果中取出 'train' 拆分，得到一个标准的Dataset对象
dataset = raw_datasets['train']

# 现在，打印一下数据集的信息和第一条样本看看结构
print("数据集结构:", dataset)
print("\n数据集列名:", dataset.column_names) # 应该输出 ['instruction', 'input', 'output', 'type']
print("\n第一条样本:")
print(dataset[0])

Generating train split: 6911 examples [00:00, 112505.13 examples/s]

数据集结构: Dataset({
    features: ['instruction', 'output', 'input', 'type'],
    num_rows: 6911
})

数据集列名: ['instruction', 'output', 'input', 'type']

第一条样本:
{'instruction': '请续写下面的诗句', 'output': '须臾走向天上来，逐却残星赶却月。', 'input': '欲出未出光辣达，千山万山如火发。', 'type': '续写'}


In [ ]:
def process_func(example):
    MAX_LENGTH = 512  # 或根据您的模型调整

    # 直接使用结构化字段，无需字符串分割！
    input_text = example['input']
    instruction_text = example['instruction']
    output_text = example['output']

    # 1. 编码输入部分（系统提示 + 用户指令和输入）
    input_prompt = f"<|im_start|>system\n现在你是一位古代诗人，请根据要求创作或续写诗句。<|im_end|>\n<|im_start|>user\n{instruction_text}：{input_text}<|im_end|>\n<|im_start|>assistant\n"
    input_ids = tokenizer(
        input_prompt,
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=False
    )["input_ids"]

    # 2. 编码输出部分（模型回复）
    response_ids = tokenizer(
        output_text,
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=False
    )["input_ids"]

    # 3. 拼接并创建labels
    # 将input_ids部分在labels中标记为-100（忽略损失），只计算response_ids的损失
    labels = [-100] * len(input_ids) + response_ids
    # 将input_ids和response_ids拼接成完整的输入序列
    input_ids = input_ids + response_ids

    # 4. 如果序列过长，进行截断
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]

    # 5. 创建注意力掩码
    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [33]:
# 应用处理函数
tokenized_dataset = dataset.map(
    process_func,
    remove_columns=dataset.column_names # 移除原始列，只保留处理后的'input_ids', 'attention_mask', 'labels'
)

Map: 100%|██████████| 6911/6911 [00:05<00:00, 1173.75 examples/s]


In [53]:
# 微调前推理
# 进行模型推理查看效果

# 确保模型在计算设备上 (NPU)
#model = model.npu() # 或者使用 .to('npu')，根据您的环境调整

# 定义提示词
prompt = "写一首关于春天的诗"

# 使用分词器的聊天模板构建输入（这是现代模型的标准做法）
inputs = tokenizer.apply_chat_template([
    {"role": "system", "content": "现在你是一位博学的古代诗人，善于创作各种题材的诗词。"},
    {"role": "user", "content": prompt}
], 
    add_generation_prompt=True,  # 在最后添加一个提示，告诉模型该它说话了
    tokenize=True,
   # return_tensors="ms",         # 返回MindSpore张量 (因您环境而定)
    return_dict=True
)#.to('npu') # 将输入也移动到NPU上

# 设置生成参数
gen_kwargs = {
    "max_length": 500,   # 生成的最大长度
    "do_sample": True,   # 使用采样而非贪婪搜索，使生成更有创意
    "top_k": 50,         # 使用top-k采样
    "temperature": 0.8,   # 温度参数，控制随机性 (0.1-1.0之间)
}

# 执行推理生成
import mindspore as ms
# with ms.no_grad(): # 在推理时不计算梯度
outputs = model.generate(**inputs, **gen_kwargs)
# 只取新生成的部分，跳过输入提示词
generated_tokens = outputs[:, inputs['input_ids'].shape[1]:] 
# 将生成的token解码为文本
response = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

print("用户提问:", prompt)
print("\n=== 模型生成结果 ===")
print(response)

AttributeError: 'list' object has no attribute 'shape'